# AML ENTERPRISE — COMPLETE MULTI-DATASET PIPELINE
OWS & EDQ Name Screening · Risk & Compliance
---
1. Setup  2. DatasetRegistry  3. InvertedIndex  4. RuleEngine  5. Orchestrator
6. Individual Screening  7. Entity Screening  8. Vessel Screening
9. Batch Screening  10. ML Training  11. Graph  12. KPIs  13. Audit

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
print('Setup complete')

In [ ]:
from data_layer.registry.dataset_registry import DatasetRegistry
registry = DatasetRegistry()
registry.load_all(n_individuals=5000, n_entities=3000, n_pep=2000,
    n_adverse_media=3000, n_vessels=1000, n_aircraft=500, n_internal_wl=1500,
    seed=42, verbose=True)
print()
print(registry.summary())

In [ ]:
from blocking_engine.inverted_index import MultiDatasetIndex
multi_idx = MultiDatasetIndex()
multi_idx.build_from_registry(registry, verbose=True)

In [ ]:
from matching_engine.rules.base_rule import MatchingRuleEngine
rule_engine = MatchingRuleEngine()
for code, rule in rule_engine._rules.items():
    print(f'  {code:<25} -> {rule.rule_name}')

In [ ]:
from orchestration.orchestrator import ScreeningOrchestrator, ScreeningInput
orchestrator = ScreeningOrchestrator(
    registry=registry, multi_index=multi_idx, rule_engine=rule_engine,
    score_threshold=0.40, max_candidates_per_dataset=300)
print('Orchestrator ready.')

In [ ]:
# INDIVIDUAL SCREENING
result = orchestrator.screen(ScreeningInput(
    name='MOHAMMED ALI HUSSAIN', entity_type='INDIVIDUAL',
    dob='1968-03-22', country='IRAN', id_number='P7654321', job_type='INDIVIDUAL_JOB'))
print(f'Decision: {result.overall_decision}  Score: {result.overall_score}')
print(f'Datasets screened: {result.datasets_screened}')
print(f'Latency: {result.latency_ms} ms')
if result.top_result and result.top_result.hits:
    print('\nTop hits:')
    for hit in sorted(result.top_result.hits, key=lambda h: h.rule_score, reverse=True)[:8]:
        print(f'  [{hit.dataset_code}] {hit.primary_name[:35]:<35} score={hit.rule_score:.3f} auto={hit.auto_alert}')

In [ ]:
# ENTITY SCREENING
result_ent = orchestrator.screen(ScreeningInput(
    name='GLOBAL PETROLEUM TRADING LTD', entity_type='ENTITY',
    country='IRAN', id_number='IR1234567', job_type='ENTITY_JOB'))
print(f'Entity Decision: {result_ent.overall_decision}  Score: {result_ent.overall_score}')
print(f'Datasets screened: {result_ent.datasets_screened}')

In [ ]:
# VESSEL SCREENING
result_vsl = orchestrator.screen(ScreeningInput(
    name='PACIFIC STAR', entity_type='VESSEL',
    imo_number='IMO9876541', job_type='VESSEL_JOB'))
print(f'Vessel Decision: {result_vsl.overall_decision}  Score: {result_vsl.overall_score}')
print(f'Auto-alert: {result_vsl.auto_alert}')

In [ ]:
# BATCH SCREENING (50 records from OFAC_SDN)
ind_df = registry.get('OFAC_SDN', entity_type='INDIVIDUAL')
batch_inputs = [ScreeningInput(
    name=row['primary_name'], entity_type='INDIVIDUAL',
    dob=row.get('dob',''), country=row.get('nationality',''),
    id_number=row.get('id_number',''), job_type='INDIVIDUAL_JOB')
    for _, row in ind_df.head(50).iterrows()]
print(f'Screening {len(batch_inputs)} records...')
batch_outputs = orchestrator.screen_batch(batch_inputs, max_workers=4, verbose=True)
summary = orchestrator.batch_summary(batch_outputs)
print('\nBatch Summary:')
for k, v in summary.items(): print(f'  {k:<32}: {v}')

In [ ]:
# ML MODEL TRAINING
from feature_engine.feature_builder import build_features, features_to_list
from ml_engine.model import AMLModel
X_rows, y_rows = [], []
all_ind = registry.get_all_for_entity_type('INDIVIDUAL')
for output in batch_outputs:
    if output.top_result and output.top_result.top_hit:
        hit = output.top_result.top_hit
        mr = {'composite_score': hit.name_score, 'score_token_sort': hit.name_score,
              'score_token_set': hit.name_score, 'score_partial': hit.name_score,
              'score_jaro_winkler': hit.name_score, 'matched_on_alias': int(hit.matched_on_alias)}
        rows = all_ind[all_ind['entity_id'] == hit.entity_id]
        if not rows.empty:
            cand = rows.iloc[0]
            feat = build_features(mr, {'name': output.input.name, 'dob': output.input.dob, 'country': output.input.country}, cand)
            X_rows.append(features_to_list(feat))
            y_rows.append(int(cand.get('risk_label', 0)))
if X_rows:
    X, y = np.array(X_rows), np.array(y_rows)
    print(f'Training: {X.shape}, positives: {y.sum()}')
    model = AMLModel(n_estimators=200, calibrate=False)
    model.train(X, y)
else:
    print('No samples — increase batch size')

In [ ]:
# GRAPH ANALYSIS
from graph_engine.graph import build_graph, graph_summary, propagate_risk
ind_df2 = registry.get_all_for_entity_type('INDIVIDUAL')
G = build_graph(ind_df2.head(500), add_country_edges=True, add_alias_edges=True)
G = propagate_risk(G, iterations=3)
for k, v in graph_summary(G).items(): print(f'  {k:<25}: {v}')

In [ ]:
# KPIs
from monitoring.kpi import confusion_matrix_metrics
from workflow.alert import alert_summary
tp=fp=tn=fn=0
all_ind2 = registry.get_all_for_entity_type('INDIVIDUAL')
for output in batch_outputs:
    gt=0
    if output.top_result and output.top_result.top_hit:
        rows = all_ind2[all_ind2['entity_id']==output.top_result.top_hit.entity_id]
        if not rows.empty: gt = int(rows.iloc[0].get('risk_label',0))
    pp = output.overall_decision in ('ALERT','REVIEW')
    if pp and gt==1: tp+=1
    elif pp and gt==0: fp+=1
    elif not pp and gt==0: tn+=1
    else: fn+=1
print('KPIs:')
for k,v in confusion_matrix_metrics(tp,fp,tn,fn).items(): print(f'  {k:<15}: {v}')
print('\nAlerts:')
for k,v in alert_summary().items(): print(f'  {k:<20}: {v}')

In [ ]:
# AUDIT EXPORT
from governance.audit import audit_log_to_df
audit_df = audit_log_to_df()
print(f'Audit entries: {len(audit_df)}')
if not audit_df.empty: print(audit_df['decision'].value_counts())
audit_df.head(5)